In [1]:
import itertools
import pickle
import math

import lightgbm
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, plot_tree
import sklearn.ensemble

In [2]:
import lightgbm
import shap

# train an XGBoost model
# X, y = shap.datasets.boston()
data = sklearn.datasets.load_iris(as_frame=True)

In [3]:
df_data = data["frame"].loc[lambda df: df.target < 3].sample(100).reset_index(drop=True)

In [4]:
df_data.head(2)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,6.7,3.1,5.6,2.4,2
1,6.4,2.8,5.6,2.2,2


In [5]:
X, y = df_data.drop(columns='target'), df_data['target']

In [6]:
y.unique()

array([2, 1, 0])

In [7]:
# model = lightgbm.LGBMModel(n_estimators=2, objective='binary', max_depth=2).fit(X, y)

model = lightgbm.LGBMModel(n_estimators=2, objective='multiclass', max_depth=2, num_classes=3).fit(X, y)
model = lightgbm.LGBMModel(n_estimators=2, objective='multiclassova', max_depth=2, num_classes=3).fit(X, y)

* For binary/two-class classifier, the raw_score is confirmed to be log odds.
* For multiclassova classifier, the raw_score is also confirmed to be log odds.
* For multiclass classifier, the raw_score is confirmed to be exponentials, which are converted to probabilites via softmax.

In [10]:
model.booster_.save_model('/tmp/lgb.txt')

In [11]:
model.predict(X[:5])

array([[0.2269135 , 0.29993297, 0.47315354],
       [0.2269135 , 0.29993297, 0.47315354],
       [0.22694077, 0.29992239, 0.47313685],
       [0.22772472, 0.49992169, 0.27235358],
       [0.42191426, 0.30344031, 0.27464543]])

In [13]:
model.predict(X[:5], raw_score=True)

array([[-1.45478028, -1.17579015, -0.71992922],
       [-1.45478028, -1.17579015, -0.71992922],
       [-1.45462484, -1.17579015, -0.71992922],
       [-1.45478028, -0.66846636, -1.27581666],
       [-0.84649545, -1.17611268, -1.27581666]])

In [19]:
exponentals = np.exp(model.predict(X[:5], raw_score=True))

In [22]:
exponentals.sum(axis=1)

array([1.02881342, 1.02881342, 1.02884971, 1.02514847, 1.01659387])

In [24]:
exponentals

array([[0.23345165, 0.30857506, 0.48678671],
       [0.23345165, 0.30857506, 0.48678671],
       [0.23348794, 0.30857506, 0.48678671],
       [0.23345165, 0.51249396, 0.27920286],
       [0.42891546, 0.30847556, 0.27920286]])

In [29]:
(exponentals.T / (exponentals.sum(axis=1))).T

array([[0.2269135 , 0.29993297, 0.47315354],
       [0.2269135 , 0.29993297, 0.47315354],
       [0.22694077, 0.29992239, 0.47313685],
       [0.22772472, 0.49992169, 0.27235358],
       [0.42191426, 0.30344031, 0.27464543]])

In [177]:
odds = np.exp(model.predict(X[:5], raw_score=True))

In [178]:
odds / (1 + odds)

array([[0.46127737, 0.26086957, 0.28499583],
       [0.27797515, 0.43145594, 0.28938765],
       [0.26896826, 0.43145594, 0.29796244],
       [0.26896826, 0.43145594, 0.38095239],
       [0.26896826, 0.43145594, 0.29796244]])